In [2]:
# dari /data 
# merge jadi 1 secara urut

In [1]:
import pandas as pd
from pathlib import Path

# 1. Definisikan folder letak file target
data_dir = Path(r"D:\RnD_JIAT\data")

# 2. Tarik semua file excel dan urutkan namanya
excel_files = sorted(data_dir.glob("*.xlsx"))

print(f"Menemukan {len(excel_files)} File:")
for file in excel_files:
    print(f"- {file.name}")

# 3. Iterasi Membaca File dan Menyimpannya (Appending)
df_list = []
for file in excel_files:
    df_temp = pd.read_excel(file)
    df_list.append(df_temp)

# 4. Satukan (Merge) secara vertikal
df_merged = pd.concat(df_list, ignore_index=True)

# 5. SORTING MUTLAK KRONOLOGIKAL
# (Sistem logger biasanya menyimpan Waktu / Timestamp pada Kolom Pertama)
kolom_waktu = df_merged.columns[0] 

# Mengkonversi ke Datetime agar Pandas bisa mensortir matematika kalender
df_merged[kolom_waktu] = pd.to_datetime(df_merged[kolom_waktu])

# Mengurutkan dari Maret -> April
df_merged = df_merged.sort_values(by=kolom_waktu).reset_index(drop=True)

# 6. OUTPUT & PENGECEKAN
print("\n=== STATUS DATA GABUNGAN ===")
print(f"Total Baris Data    : {len(df_merged)} baris")
print(f"Waktu Paling Awal   : {df_merged[kolom_waktu].min()}")
print(f"Waktu Paling Akhir  : {df_merged[kolom_waktu].max()}")

# Preview akhir
display(df_merged.head())
display(df_merged.tail())

Menemukan 3 File:
- Pos AWLR JIAT Pondok Kahuru - Rerata Muka Air Tanah - 2026-03-01 00_00 - 2026-03-07 23_00.xlsx
- Pos AWLR JIAT Pondok Kahuru - Rerata Muka Air Tanah - 2026-03-08 00_00 - 2026-03-31 23_00.xlsx
- Pos AWLR JIAT Pondok Kahuru - Rerata Muka Air Tanah - 2026-04-01 00_00 - 2026-04-13 10_00.xlsx

=== STATUS DATA GABUNGAN ===
Total Baris Data    : 934 baris
Waktu Paling Awal   : 2026-03-01 00:00:00
Waktu Paling Akhir  : 2026-04-13 10:00:00


,Waktu,Rerata Muka Air Tanah,Minimal,Maksimal
0,2026-03-01 00:00:00,17.502,17.49,17.51
1,2026-03-01 01:00:00,17.490,17.49,17.50
2,2026-03-01 02:00:00,17.479,17.47,17.49
3,2026-03-01 03:00:00,17.470,17.47,17.48
4,2026-03-01 04:00:00,17.510,17.47,17.56


,Waktu,Rerata Muka Air Tanah,Minimal,Maksimal
929,2026-04-13 06:00:00,18.082,18.03,18.11
930,2026-04-13 07:00:00,18.145,18.12,18.16
931,2026-04-13 08:00:00,18.162,18.16,18.17
932,2026-04-13 09:00:00,18.123,18.10,18.16
933,2026-04-13 10:00:00,18.090,18.09,18.09


In [ ]:
# 1. Definisikan jalur penyimpanan (Disarankan langsung kembalikan ke folder /data)
jalur_simpan = r"D:\RnD_JIAT\data\Pos_AWLR_JIAT_Pondok_Kahuru.csv"

# 2. Eksekusi penyimpanan ke CSV
# (index=False digunakan agar nomor baris bawaan Pandas 0, 1, 2 tidak ikut menyusup masuk ke kolom)
# df_merged.to_csv(jalur_simpan, index=False)

# 3. Notifikasi Sukses
print(f"✅ Dataframe berhasil diekspor menjadi CSV tunggal!")
print(f"📁 Lokasi: {jalur_simpan}")

NameError: name 'df_merged' is not defined

In [2]:
import pandas as pd
import plotly.express as px

# =======================================================
# 1. IMPORT & FILTER KOLOM
# =======================================================
file_path = r"D:\RnD_JIAT\data\Pos_AWLR_JIAT_Pondok_Kahuru.csv"
df_plot = pd.read_csv(file_path)

# Deteksi Kolom Waktu dan Tetapkan formatnya secara mutlak menjadi Datetime
kolom_waktu = df_plot.columns[0]
df_plot[kolom_waktu] = pd.to_datetime(df_plot[kolom_waktu])

# NYALAKAN FILTER: Hanya sedot memori kolom data yang mengandung kata "Rerata"
kolom_rerata_saja = [col for col in df_plot.columns if 'rerata' in col.lower()]

# Langkah aman cadangan (Jika kata 'Rerata' tidak ditemukan, paksa plotting kolom kedua)
if len(kolom_rerata_saja) == 0:
    kolom_rerata_saja = [df_plot.columns[1]]

# =======================================================
# 2. EKSEKUSI VISUALISASI DENGAN PLOTLY
# =======================================================
fig = px.line(
    df_plot, 
    x=kolom_waktu, 
    y=kolom_rerata_saja,
    title='<b>Grafik Rerata Muka Air Tanah (AWLR) - JIAT Pondok Kahuru</b>',
    template='plotly_dark'
)

# =======================================================
# 3. KOSMETIKA & LAYOUTING INTERAKTIF
# =======================================================
fig.update_layout(
    xaxis_title='Tanggal Observasi (Timeline)',
    yaxis_title='Kedalaman (Meter)',
    hovermode='x unified',             # Kunci agar kursor mouse memunculkan toolstip panjang menyambung
    legend_title='Indikator Parameter',
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1
    )
)

# [OPSIONAL] Karena ini membahas Kedalaman sumur, biasanya hidrolog membalik sumbu Y
# Semakin bertambah nilai Kedalamannya, grafiknya akan semakin turun. 
# Jika ingin memberlakukan standar tersebut, HAPUS TANDA PAGAR (#) di awal baris kode ini:
# fig.update_yaxes(autorange="reversed")

# Render Diagram!
fig.show()

In [3]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Pastikan df_merged sudah di-loading
df_merged = df_plot
kolom_waktu = df_merged.columns[0]
df_eda = df_merged.copy()
df_eda.set_index(kolom_waktu, inplace=True)

# 1. RESAMPLING UNTUK MENDETEKSI ABSEN" (Missing Values)
# Perintah ini akan memaksa rentang data kalender yang "bolong" dimunculkan kembali dengan isi Kosong (NaN)
df_eda = df_eda.resample('1H').mean()

# 2. FEATURE ENGINEERING (Ekstraksi Waktu Harian & Muluskan Trend)
df_eda['Jam_Operasi'] = df_eda.index.hour
df_eda['Trend_Jangka_Panjang'] = df_eda['Rerata Muka Air Tanah'].rolling(window=24, min_periods=1).mean()
df_eda['Volatilitas_Sensor'] = df_eda['Maksimal'] - df_eda['Minimal']


# =========================================================================
# (GRAFIK 1): TREN BASELINE DEFISIT AKUIFER & LOKASI PEMADAMAN SENSOR (GAP)
# =========================================================================
fig1 = go.Figure()
# Fluktuasi Harian (Warna Biru transparan)
fig1.add_trace(go.Scatter(x=df_eda.index, y=df_eda['Rerata Muka Air Tanah'], 
                          mode='lines', name='Raw Data TMA', line=dict(color='#0ea5e9', width=1), opacity=0.4))
# Tren Dasar Pendarahan Sumur (Warna Kuning terang)
fig1.add_trace(go.Scatter(x=df_eda.index, y=df_eda['Trend_Jangka_Panjang'], 
                          mode='lines', name='Tren Akuifer Kritis', line=dict(color='#facc15', width=3)))

fig1.update_layout(title='<b>EDA 1: Tren Bukti Defisit Jangka Panjang & Area Sensor Offline</b>', 
                   template='plotly_dark', hovermode='x unified')
fig1.update_yaxes(autorange='reversed') # Wajib Dibalik agar "Lebih dalam" = "Kebawah"
fig1.show()


# =========================================================================
# (GRAFIK 2): BOXPLOT ANALISIS PERILAKU POMPA & KONSUMSI SETIAP JAM 
# =========================================================================
fig2 = px.box(df_eda, x='Jam_Operasi', y='Rerata Muka Air Tanah', 
              color_discrete_sequence=['#fb7185'], 
              title='<b>EDA 2: Kapan Sumur Paling Rawan Cekik? (Sebaran Kedalaman per-Jam)</b>',
              template='plotly_dark')

fig2.update_layout(xaxis_title='Waktu Jam Operasional (00:00 - 23:00)', yaxis_title='Kedalaman Median (M)')
fig2.update_yaxes(autorange='reversed')
fig2.show()


# =========================================================================
# (GRAFIK 3): DETEKSI ANOMALI GUNCANGAN AIR (TURBULENSI PADA PIPA SUMUR)
# =========================================================================
fig3 = px.bar(df_eda, x=df_eda.index, y='Volatilitas_Sensor',
              title='<b>EDA 3: Deteksi Gejolak Air (Beda Angka Max & Min per Jam)</b>',
              template='plotly_dark', color_discrete_sequence=['#a78bfa'])

fig3.update_layout(xaxis_title='Tanggal Observasi', yaxis_title='Jarak Gejolak (Delta m)')
fig3.show()

C:\Users\fadel\AppData\Local\Temp\ipykernel_4332\4062702287.py:13: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [6]:

# =========================================================================
# 2. FEATURE ENGINEERING (Penyatuan dengan Variabel Spesifikasi Sumur Nyata)
# =========================================================================
df_eda['Jam_Operasi'] = df_eda.index.hour 
df_eda['Trend_Jangka_Panjang'] = df_eda['Rerata Muka Air Tanah'].rolling(window=24, min_periods=1).mean()
df_eda['Volatilitas_Sensor'] = df_eda['Maksimal'] - df_eda['Minimal']

# --- [BAGIAN BARU] INPUT KONSTANTA POS AWLR ---------
KEDALAMAN_SUMUR = 70.0   # Jarak galian dari permukaan atas sumur ke mentok dasar (m)
KEDALAMAN_SENSOR = 60.0  # Jarak alat sensor Logger diturunkan ke bawah/terendam (m)

# 1. Menghitung 'Data Air Tanah' yang Anda Maksud (Total tinggi cadangan SISA air di dlm corong dari dasar)
df_eda['Data_Air_Tanah'] = KEDALAMAN_SUMUR - df_eda['Rerata Muka Air Tanah']

# 2. Menghitung Head Presur Sensor (Tinggi/beban sisa air MURNI di atas alat logger)
# Fungsinya: jika air drop ke kedalaman 61m, logger akan otomatis mengukur angka minus/Nol (Kering)
df_eda['Pembacaan_Murni_Sensor'] = KEDALAMAN_SENSOR - df_eda['Rerata Muka Air Tanah']

# =========================================================================
# (GRAFIK 4/OPSI BARU): VISUALISASI KRITIS SISA KETINGGIAN AIR TANAH ASLI
# =========================================================================
fig4 = go.Figure()

# Karena yg diukur sekarang "Banyaknya air", maka grafiknya tidak perlu dibalik.
# (Makin kurvanya terjun bebas, makin sedikit air yang tersisa di dasar sumur)
fig4.add_trace(go.Scatter(
    x=df_eda.index, y=df_eda['Data_Air_Tanah'], 
    mode='lines', name='Total Sisa Volume Tampung (Data Air Tanah)', 
    line=dict(color='#8b5cf6', width=2), fill='tozeroy', fillcolor='rgba(139, 92, 246, 0.2)'
))

# Kita tambahkan GARIS BATAS BAHAYA (Posisi alat Sensor Logger AWLR-nya mengintip)
# Sensor ada di kedalaman 60m, jika sumur 70m, berarti sensor diletakkan HANYA 10 Meter dari dasar!
batas_kritis_dasar = KEDALAMAN_SUMUR - KEDALAMAN_SENSOR
fig4.add_hline(y=batas_kritis_dasar, line_dash="dash", line_color="#ef4444", line_width=2,
               annotation_text=f"Batas Terbawah/Blindspot Sensor (Level {batas_kritis_dasar}m dari dasar)")

fig4.update_layout(
    title='<b>EDA 4: Pantauan Deplesi Stok Cadangan Air Sebenarnya Pada Pipa Sumur Induk</b>',
    xaxis_title='Tanggal Perekaman Kalender',
    yaxis_title='Ketinggian Sisa Air dari Dasar (Meter)',
    template='plotly_dark', hovermode='x unified'
)

fig4.show()


In [20]:
# =========================================================================
# 2. FEATURE ENGINEERING (Kalkulasi Berbasis Kolom Rerata Dinamis)
# =========================================================================
df_eda['Jam_Operasi'] = df_eda.index.hour

# Ekstrak secara eksak nama kolom rill-nya agar tidak KeyError
kolom_target_rerata = kolom_rerata_saja[0] 

# Tren dihitung secara otomatis memakai nama kolom tsb
df_eda['Trend_Jangka_Panjang'] = df_eda[kolom_target_rerata].rolling(window=24, min_periods=1).mean()
df_eda['Volatilitas_Sensor'] = df_eda['Maksimal'] - df_eda['Minimal']

# --- KONSTANTA POS AWLR ---------
KEDALAMAN_SUMUR = 70.0   # (Meter)
KEDALAMAN_SENSOR = 60.0  # (Meter)

# 1. Menghitung "DATA AIR TANAH" (Selisih mutlak Kedalaman dengan nilai TMA rerata)
df_eda['Data_Air_Tanah'] = KEDALAMAN_SUMUR - df_eda[kolom_target_rerata]

# 2. Menghitung Head Presur Sensor
df_eda['Pembacaan_Murni_Sensor'] = KEDALAMAN_SENSOR - df_eda[kolom_target_rerata]

# =========================================================================
# (GRAFIK 1: REVISI): FOKUS MURNI PADA POLA SIKLUS HARIAN
# =========================================================================
fig1 = go.Figure()

# Kita plot garis fluktuasi aslinya (*Raw Data*) dengan warna Solid (Tidak Transparan lagi)
# Agar patahan lembah siklus nyala-matinya terlihat amat menonjol & tajam
fig1.add_trace(go.Scatter(
    x=df_eda.index, 
    y=df_eda[kolom_target_rerata], 
    mode='lines', 
    name='Siklus Fluktuasi TMA', 
    line=dict(color='#0ea5e9', width=2)
))

fig1.update_layout(
    title='<b>EDA 1: Pemantauan Siklus Tarikan (Drawdown) Harian Akuifer</b>', 
    xaxis_title='Timeline Observasi AWLR',
    yaxis_title='Kedalaman Muka Air (Meter)',
    template='plotly_dark', 
    hovermode='x unified',
    showlegend=True
)

# Balik arah Sumbu Y (Prinsip Hidrologi: Makin banyak angka meter = Makin dalam terjun ke dasar)
fig1.update_yaxes(autorange='reversed') 

fig1.show()


In [ ]:
df_eda

In [ ]:
# ==============================================================================
# MENGHITUNG DINAMIS RANGE BERDASARKAN MIN/MAX DATA (+ MARGIN 10%)
# ==============================================================================
min_tma = df_eda[kolom_mat_rerata].min()
max_tma = df_eda[kolom_mat_rerata].max()

# Beri jarak ruang napas (margin offset) 10% di atas dan bawah grafik
margin_tma = (max_tma - min_tma) * 0.10
if margin_tma == 0: 
    margin_tma = 1.0 # Antisipasi jika data sensor stagnan sempurna

# Batas Y-Axis Kiri (TMA) - Dibuat terbalik: Angka besar (dalam) di bawah
tma_batas_bawah = max_tma + margin_tma 
tma_batas_atas = min_tma - margin_tma  

# Batas Y-Axis Kanan (Data Air) - Normal: Angka kecil di bawah
data_batas_bawah = KEDALAMAN_SUMUR - tma_batas_bawah
data_batas_atas = KEDALAMAN_SUMUR - tma_batas_atas

# ==============================================================================
# KOSMETIKA DASHBOARD KHAS DATA SCIENCE
# ==============================================================================
fig_dual.update_layout(
    title='<b>Analisis Cermin: Fluktuasi Turunnya Ketinggian Air Tanah (TMA vs Riil Volume)</b>',
    xaxis_title='Waktu Log Kalender',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Custom Sumbu Kiri (TMA) - Dinamis dari Min/Max Data
fig_dual.update_yaxes(
    title_text="Kedalaman Air Muka (Meter)", 
    range=[tma_batas_bawah, tma_batas_atas], # Otomatis reversed karena batas_bawah nilainya lebih besar
    secondary_y=False,
    color="#0ea5e9",
    showgrid=False
)

# Custom Sumbu Kanan (Data Air Tanah Riil Volumetrik) - Dinamis dari Min/Max Data
fig_dual.update_yaxes(
    title_text="Sisa Ketinggian Tiang Air (Meter)", 
    range=[data_batas_bawah, data_batas_atas], 
    secondary_y=True,
    color="#d946ef",
    showgrid=True,
    gridcolor='rgba(255, 255, 255, 0.1)'
)

fig_dual.show() # REF011

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. SETUP & HITUNG DELTA
# ==============================================================================

# Pastikan tidak merusak df_eda utama

df_calc = df_eda.copy()
df_calc['Waktu'] = df_calc.index # Menyatukan index datetime ke dalam kolom untuk gampang dianalisis

# Menghitung Delta (Perubahan Tinggi Air Tanah) per baris / jam
df_calc['Delta_Volume'] = df_calc['Data_Air_Tanah'].diff()

# ==============================================================================
# 2. DEFINISI STATUS AKUIFER (Discharge, Charge, Idle)
# ==============================================================================
THRESHOLD = 0.02 # Margin toleransi 2 cm (0.02 m). Hindari noise riak air kecil agar tidak dibaca sbg siklus

def tentukan_status(delta):
    if pd.isna(delta):
        return 'Unknown'
    elif delta <= -THRESHOLD:
        return 'Discharge' # Pompa nenyedot (Volume berkurang)
    elif delta >= THRESHOLD:
        return 'Charge'    # Alam mengisi ulang (Volume bertambah / Recovery)
    else:
        return 'Idle'      # Stabil / Mentok (Tidak ada penurunan/kenaikan berarti)

df_calc['Status'] = df_calc['Delta_Volume'].apply(tentukan_status)

# ==============================================================================
# 3. CLUSTERING ID SIKLUS BERANTAI
# ==============================================================================
# Setiap kali status berubah (misal dari "Charge" ke "Discharge"), ID siklus bergeser (tambah 1).
# Ini trik vital mendeteksi 1 Event Pompa (dari awal nyala sampai mati).
df_calc['Status_Shift'] = df_calc['Status'] != df_calc['Status'].shift(1)
df_calc['Cycle_ID'] = df_calc['Status_Shift'].cumsum()

# ==============================================================================
# 4. AGREGRASI DAN DURASI SIKLUS
# ==============================================================================
cycle_stats = df_calc.groupby(['Cycle_ID', 'Status']).agg(
    Waktu_Mulai=('Waktu', 'min'),
    Waktu_Selesai=('Waktu', 'max'),
    Durasi=('Waktu', lambda x: x.max() - x.min()),
    Total_Meter=('Data_Air_Tanah', lambda x: abs(x.iloc[-1] - x.iloc[0])) # Seberapa banyak air kepompa/terisi
).reset_index()

# Bila durasi 0 (karena kejadian sangat cepat di logger / hanya 1 row), kita amankan jadi minimum 1 jam
cycle_stats.loc[cycle_stats['Durasi'] == pd.Timedelta(seconds=0), 'Durasi'] = pd.Timedelta(hours=1)

# Pisahkan list Log khusus Charge dan Khusus Discharge
cycles_charge = cycle_stats[cycle_stats['Status'] == 'Charge'].copy()
cycles_discharge = cycle_stats[cycle_stats['Status'] == 'Discharge'].copy()

# ==============================================================================
# 5. MENGHITUNG STATISTIK RATA-RATA & SIKLUS PER HARI
# ==============================================================================
df_calc['Tanggal'] = df_calc.index.date
siklus_per_hari = df_calc.groupby('Tanggal').apply(
    lambda x: len(x[x['Status'] == 'Discharge']['Cycle_ID'].unique())
).reset_index(name='Total_Discharge')

mean_siklus_harian = siklus_per_hari['Total_Discharge'].mean()
mean_recovery_time = cycles_charge['Durasi'].mean()

# ==============================================================================
# 6. MENGHITUNG ETR (ESTIMATED TIME OF RECOVERY) KONDISI TERKINI
# ==============================================================================
# Untuk memproyeksikan kapan pulih, kita harus tahu "Kecepatan Isi Alamiah" akuifer tersebut.
total_meter_recovered = cycles_charge['Total_Meter'].sum()
total_jam_recovered = cycles_charge['Durasi'].dt.total_seconds().sum() / 3600

kecepatan_recovery_perjam = total_meter_recovered / total_jam_recovered if total_jam_recovered > 0 else 0

# Titik Acuan ETR
statis_ideal = df_calc['Data_Air_Tanah'].max() # Kapasitas penuh (Titik Idle termaksimal yg tercatat)
volume_saat_ini = df_calc['Data_Air_Tanah'].iloc[-1]
defisit_meter = statis_ideal - volume_saat_ini # Berapa meter lagi yg kurang?

if kecepatan_recovery_perjam > 0 and defisit_meter > 0:
    etr_jam = defisit_meter / kecepatan_recovery_perjam
else:
    etr_jam = 0.0 # Sudah Penuh / Terjadi Anomali

# ==============================================================================
# 7. CETAK HASIL (REPORTING)
# ==============================================================================
print("="*70)
print("📊 LAPORAN PERILAKU SIKLUS AKUIFER (HYDROLOGICAL ANALYTICS) 📊")
print("="*70)
print(f"1. Total Kejadian Pumping (Disch) : {len(cycles_discharge)} Event (Siklus Tarikan)")
print(f"2. Total Kejadian Pemulihan (Recv)  : {len(cycles_charge)} Event (Siklus Isi Ulang)")
print(f"3. Rata-rata Siklus Pompa per Hari  : {mean_siklus_harian:.1f} kali pompa nyala per hari")
print(f"4. Mean Recovery Time (Historis)    : {mean_recovery_time} (Wata-rata waktu tiap selesai pompa)")
print(f"5. Kecepatan Alamiah Pengisian      : {kecepatan_recovery_perjam:.3f} Meter/Jam\n")

print("🔍 STATUS REAL-TIME (PROYEKSI AKHIR DATA):")
print(f"   Level Air Tanah Saat Ini         : {volume_saat_ini:.2f} Meter dari Batas Bawah")
print(f"   Level Air Statis (Potensial Maks): {statis_ideal:.2f} Meter")
print(f"   Defisit Air (Drawdown/Kekosongan): {defisit_meter:.2f} Meter")
print(f"   ETR (Est. Time of Recovery)      : {etr_jam:.1f} Jam (Dari saat ini untuk kembali ke wujud 100%)\n")

print("Rekap 5 Riwayat Isi Ulang (Charge/Recovery) Historis Terakhir:")
display(cycles_charge.tail(5)[['Waktu_Mulai', 'Waktu_Selesai', 'Durasi', 'Total_Meter']].reset_index(drop=True))

In [26]:
import plotly.graph_objects as go
import pandas as pd

fig_cycle = go.Figure()

# ==============================================================================
# 1. PLOT GARIS UTAMA (SISA KETINGGIAN AIR)
# ==============================================================================
fig_cycle.add_trace(
    go.Scatter(
        x=df_calc['Waktu'],
        y=df_calc['Data_Air_Tanah'],
        mode='lines',
        name='Ketinggian Air Aktual',
        line=dict(color='#0ea5e9', width=3)
    )
)

# ==============================================================================
# 2. HIGHLIGHT AREA ZONA POMPA (RED) & ZONA ISI ULANG (GREEN)
# ==============================================================================
# Mewarnai setiap blok latar belakang yang terdeteksi sebagai zona tarikan (Discharge)
for _, row in cycles_discharge.iterrows():
    fig_cycle.add_vrect(
        x0=row['Waktu_Mulai'], x1=row['Waktu_Selesai'],
        fillcolor="#ef4444", opacity=0.15, line_width=0 # Warna merah transparan
    )

# Mewarnai setiap blok latar belakang yang terdeteksi sebagai zona pengisian (Charge)
for _, row in cycles_charge.iterrows():
    fig_cycle.add_vrect(
        x0=row['Waktu_Mulai'], x1=row['Waktu_Selesai'],
        fillcolor="#22c55e", opacity=0.15, line_width=0 # Warna hijau transparan
    )

# ==============================================================================
# 3. TRACE DUMMY AGAR ZONA WARNA MUNCUL DI LEGEND (Keterangan)
# ==============================================================================
fig_cycle.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines', name='Zona Pompa Aktif / Sedot (Disch)',
    line=dict(color='rgba(239, 68, 68, 0.5)', width=4)
))
fig_cycle.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines', name='Zona Pengisian / Istirahat (Recv)',
    line=dict(color='rgba(34, 197, 94, 0.5)', width=4)
))

# ==============================================================================
# 4. PROYEKSI VISUAL ETR (ESTIMATED TIME OF RECOVERY) DARI TITIK TERAKHIR
# ==============================================================================
if etr_jam > 0:
    waktu_terakhir = df_calc['Waktu'].iloc[-1]
    waktu_etr = waktu_terakhir + pd.Timedelta(hours=etr_jam)
    
    # Titik awal saat ini menuju waktu prediksi masa depan (ETR)
    fig_cycle.add_trace(
        go.Scatter(
            x=[waktu_terakhir, waktu_etr],
            y=[volume_saat_ini, statis_ideal],
            mode='lines+markers',
            name=f'Proyeksi ETR ({etr_jam:.1f} Jam ke depan)',
            line=dict(color='#fbbf24', width=3, dash='dash'), # Kuning putus-putus
            marker=dict(size=10, symbol='star-diamond', color='#f59e0b')
        )
    )
    
    # Garis Horizontal Batas Atas yang menunjukkan "Kapan dia Penuh (Statis/Pulih)"
    fig_cycle.add_hline(
        y=statis_ideal, 
        line_dash="dot", 
        annotation_text=f"Level Statis Pulih 100% ({statis_ideal:.2f} m)", 
        annotation_position="top right",
        line_color="rgba(255, 255, 255, 0.4)"
    )

# ==============================================================================
# 5. KOSMETIKA DASHBOARD KHAS DATA SCIENCE
# ==============================================================================
fig_cycle.update_layout(
    title='<b>Visualisasi Forensik Siklus Akuifer: Discharge, Recovery & Proyeksi ETR</b>',
    xaxis_title='Waktu Jam Aktual (Sumbu Waktu Sejarah -> Masa Depan)',
    yaxis_title='Ketinggian Air Tanah (Meter)',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Render
fig_cycle.update_yaxes(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)')
fig_cycle.update_xaxes(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)')
fig_cycle.show()

c:\Users\fadel\anaconda3\envs\torch\Lib\site-packages\plotly\io\_json.py:560: UserWarning:

Discarding nonzero nanoseconds in conversion.



In [27]:
df_eda

,Rerata Muka Air Tanah,Minimal,Maksimal,Jam_Operasi,Trend_Jangka_Panjang,Volatilitas_Sensor,Data_Air_Tanah,Pembacaan_Murni_Sensor
Waktu,,,,,,,,
2026-03-01 00:00:00,17.502,17.49,17.51,0,17.502000,0.02,52.498,42.498
2026-03-01 01:00:00,17.490,17.49,17.50,1,17.496000,0.01,52.510,42.510
2026-03-01 02:00:00,17.479,17.47,17.49,2,17.490333,0.02,52.521,42.521
2026-03-01 03:00:00,17.470,17.47,17.48,3,17.485250,0.01,52.530,42.530
2026-03-01 04:00:00,17.510,17.47,17.56,4,17.490200,0.09,52.490,42.490
...,...,...,...,...,...,...,...,...
2026-04-13 06:00:00,18.082,18.03,18.11,6,18.047208,0.08,51.918,41.918
2026-04-13 07:00:00,18.145,18.12,18.16,7,18.047625,0.04,51.855,41.855
2026-04-13 08:00:00,18.162,18.16,18.17,8,18.048417,0.01,51.838,41.838


In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# BUKTIKAN: Lebarkan rentang data menjadi 1 Minggu untuk melihat repetisi harian
df_plot = df_eda.loc["2026-03-01":"2026-04-12"].copy()

# Tangkap identitas kolom Depth (TMA)
kolom_mat_rerata = [col for col in df_plot.columns if 'rerata' in col.lower()][0]

# ==============================================================================
# ALGORITMA WARNA FASE BERDASARKAN SELISIH KEDALAMAN (DELTA)
# ==============================================================================
# diff() akan mengukur perbedaan jarak dengan jam sebelumnya
delta_kedalaman = df_plot[kolom_mat_rerata].diff().fillna(0)

warna_fase = []
for d in delta_kedalaman:
    if d < -0.003:  # Disedot: nilai menyusut (negatif)
        warna_fase.append('#ef4444') # Warna Merah (Discharging / Pompa Aktif / Surut)
    elif d > 0.003: # Memulih: nilai bertambah (positif)
        warna_fase.append('#facc15') # Warna Kuning (Charging / Sumur Memulih Naik)
    else:
        warna_fase.append('#94a3b8') # Warna Abu-abu (Fase Istirahat / Stabil)

# Deklarasi Mode "DUAL-AXIS"
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

# ==============================================================================
# TRACE 1: MUKA AIR TANAH (TMA) - Highlight Titik Warna Tergantung Fase
# ==============================================================================
fig_dual.add_trace(
    go.Scatter(
        x=df_plot.index, 
        y=df_plot[kolom_mat_rerata], 
        name="Kedalaman Jarak Pipa TMA",
        mode="lines+markers", 
        # Kita jadikan garis utamanya samar (berperan sebagai rel saja)
        line=dict(color="rgba(255, 255, 255, 0.2)", width=2), 
        marker=dict(
            color=warna_fase,    # Suntik list array warna buatan fungsi diatas kesini!
            size=7,
            symbol='circle',
            line=dict(color='rgba(0,0,0,0.5)', width=0.5) # Border hitam memisahkan titik
        )
    ),
    secondary_y=False,
)

# ==============================================================================
# TRACE DUMMY: UNTUK MUNCULKAN INSTRUKSI LEGENDA WARNA BUKTI DI ATAS GRAFIK
# ==============================================================================
fig_dual.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='#facc15', size=10), name='Fase Pemulihan (Kuning)'),
    secondary_y=False
)
fig_dual.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='#ef4444', size=10), name='Fase Disedot Pompa (Merah)'),
    secondary_y=False
)

# ==============================================================================
# TRACE 2: DATA AIR TANAH (SISA)
# ==============================================================================
# Menyesuaikan opacity agar titik kuning/merah kita tervisualisasi lebih tajam / menonjol
fig_dual.add_trace(
    go.Scatter(
        x=df_plot.index, 
        y=df_plot['Data_Air_Tanah'] if 'Data_Air_Tanah' in df_plot.columns else df_plot[kolom_mat_rerata], 
        name="Sisa Tiang Volumetrik AIr",
        mode="lines", 
        line=dict(color="#d946ef", width=1, dash='dot'), 
        fill='tozeroy', fillcolor='rgba(217, 70, 239, 0.05)' # Opacity latar air diturunkan > 0.05
    ),
    secondary_y=True,
)

# ==============================================================================
# KOSMETIKA DASHBOARD MAKSIMUM
# ==============================================================================
fig_dual.update_layout(
    title='<b>Analisis Deteksi 7 Hari: Bukti Kuat Waktu Charging (Kuning) vs Pumping (Merah)</b>',
    xaxis_title='Siklus Jam Harian',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Render jam 12 harian spesifik di sumbu X agar waktu kelihatan rinci
fig_dual.update_xaxes(
    dtick="43200000", 
    tickformat="%d %b (%H:%M)" 
)

fig_dual.update_yaxes(
    title_text="Kedalaman Bacaan Alat (Meter)", 
    autorange="reversed", 
    secondary_y=False,
    color="#94a3b8",
    showgrid=False
)

fig_dual.update_yaxes(
    title_text="Sisa Ketinggian Tiang Air Aktual", 
    secondary_y=True,
    color="#d946ef",
    showgrid=True,
    gridcolor='rgba(255, 255, 255, 0.1)'
)

fig_dual.show()


In [19]:
# MENGHITUNG FREKUENSI DAN DURASI RECOVERY PER HARI
# =========================================================

# 1. Identifikasi mana baris yang merupakan fase Recovery secara Boolean
# (Pakai logika Kuning / delta positif seperti yang telah kita bicarakan)
delta_kedalaman = df_plot[kolom_mat_rerata].diff().fillna(0)
is_recovery = delta_kedalaman > 0.003

# 2. Ambil informasi interval data (jarak log jam antar baris dari telemetry)
interval_jam = (df_plot.index[1] - df_plot.index[0]).total_seconds() / 3600

# 3. Kelompokkan siklus agar tidak dihitung per data point, melainkan "per event".
# Trik Pandas: Menggabungkan baris-baris berdekatan/bersusun yang punya status sama 
id_blok_event = (is_recovery != is_recovery.shift()).cumsum()

# 4. Buat DataFrame khusus yang berisi data "Pemulihan" saja
df_fase_kuning = df_plot[is_recovery]

# 5. Ekstrak Laporannya berdasar Tanggal Hari 
print("=== Ringkasan Evaluasi Sumur (Pemulihan/Recovery) ===")
for tanggal, group_per_hari in df_fase_kuning.groupby(df_fase_kuning.index.date):
    
    # Menghitung berapa ID blok unik hari itu (Nilai 'n kali')
    n_kali = group_per_hari.groupby(id_blok_event).ngroups 
    
    # Total durasi = Total baris pemulihan dikali rentang interval jam datanya
    durasi_total = len(group_per_hari) * interval_jam 
    
    print(f"Hari {tanggal} - {n_kali} kali recovery, Total durasi pemulihan: {durasi_total} Jam")

=== Ringkasan Evaluasi Sumur (Pemulihan/Recovery) ===
Hari 2026-04-01 - 2 kali recovery, Total durasi pemulihan: 6.0 Jam
Hari 2026-04-02 - 3 kali recovery, Total durasi pemulihan: 9.0 Jam
Hari 2026-04-03 - 3 kali recovery, Total durasi pemulihan: 8.0 Jam
Hari 2026-04-04 - 3 kali recovery, Total durasi pemulihan: 6.0 Jam
Hari 2026-04-05 - 2 kali recovery, Total durasi pemulihan: 7.0 Jam
Hari 2026-04-06 - 4 kali recovery, Total durasi pemulihan: 8.0 Jam
Hari 2026-04-07 - 4 kali recovery, Total durasi pemulihan: 7.0 Jam
Hari 2026-04-08 - 3 kali recovery, Total durasi pemulihan: 8.0 Jam
Hari 2026-04-09 - 2 kali recovery, Total durasi pemulihan: 5.0 Jam
Hari 2026-04-10 - 3 kali recovery, Total durasi pemulihan: 8.0 Jam
Hari 2026-04-11 - 3 kali recovery, Total durasi pemulihan: 8.0 Jam
Hari 2026-04-12 - 5 kali recovery, Total durasi pemulihan: 9.0 Jam


In [ ]:
import math

# --- PARAMETER HIDROLOGI (Perlu disesuaikan dengan aktual) ---
Q_DEBIT = 150.0      # Contoh Debit Pumping (m3/hari)
STATIC_TMA = 2.0     # Muka air tanah mula-mula / batas stabil (meter)
DURASI_POMPA = 4.0   # Estimasi rata-rata waktu pompa nyala sebelum siklus pulih (t_p)
SISA_TARGET_M = 0.10 # Batas toleransi dibilang 'sudah pulih penuh' (misal kurang dari 10cm)

print("=== Hasil Prediksi Kruseman & De Ridder (Theis Method) ===")
# Lakukan iterasi setiap hari pada data pemulihan
for tanggal, group_per_hari in df_fase_kuning.groupby(df_fase_kuning.index.date):
    n_kali = group_per_hari.groupby(id_blok_event).ngroups
    print(f"\nHari {tanggal} ({n_kali} kali Pemulihan/Event):")
    
    # Lakukan pengecekan untuk Masing-Masing Kejadian di hari tersebut
    for urutan, (id_blok, blok_kuning) in enumerate(group_per_hari.groupby(id_blok_event), start=1):
        
        # Metode Theis mensyaratkan kita menganalisis titik-titik awal pemulihan.
        # Karena ini data per-jam, kita butuh seenggaknya data jam ke-1 dan jam ke-2 mati pompa.
        if len(blok_kuning) < 2:
            print(f"  - Siklus {urutan}: Gagal (butuh minimal durasi log 2 baris pasca pompa mati)")
            continue
            
        try:
            # 1. Ekstrak residual drawdown (s') di t'=1 dan t'=2 
            s_prime_1 = blok_kuning[kolom_mat_rerata].iloc[0] - STATIC_TMA
            s_prime_2 = blok_kuning[kolom_mat_rerata].iloc[1] - STATIC_TMA
            
            # Waktu (t) adalah durasi pompa + masa transisi itu (t')
            t_prime_1 = 1
            t_1 = DURASI_POMPA + t_prime_1
            t_prime_2 = 2
            t_2 = DURASI_POMPA + t_prime_2
            
            # 2. Kemiringan Logaritmik (Slope Kurva Theis) menurut Buku De Ridder Bab 13
            log_diff = math.log10(t_1 / t_prime_1) - math.log10(t_2 / t_prime_2)
            delta_s = s_prime_1 - s_prime_2
            
            if log_diff <= 0 or delta_s <= 0:
                print(f"  - Siklus {urutan}: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)")
                continue
                
            slope = delta_s / log_diff
            
            # 3. Hitung Transmisivitas (T) (Meter persegi / hari)
            transmissivity = (2.30 * Q_DEBIT) / (4 * math.pi * slope)
            
            # 4. Prediksi Jam yang dibutuhkan sisa air untuk kembali ke STATIC_TMA
            if s_prime_2 <= SISA_TARGET_M:
                 prediksi_jam = 0.0 # Sudah dianggap pulih sesuai target
            else:
                 target_log_ratio = SISA_TARGET_M / slope
                 ratio_val = 10**target_log_ratio
                 
                 # Hitung ekstensi t' menggunakan rumus balik Theis
                 t_prime_target = DURASI_POMPA / (max(0.001, ratio_val - 1))
                 prediksi_jam = max(0, t_prime_target - t_prime_2)
            
            print(f"  - Siklus {urutan}: Estimasi Butuh ±{prediksi_jam:.1f} Jam Lagi | Transmisivitas T = {transmissivity:.2f} m²/hari")
        
        except Exception as e:
             print(f"  - Siklus {urutan}: Error kalkulasi matematis ({e})")

=== Hasil Prediksi Kruseman & De Ridder (Theis Method) ===

Hari 2026-04-01 (2 kali Pemulihan/Event):
  - Siklus 1: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)
  - Siklus 2: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)

Hari 2026-04-02 (3 kali Pemulihan/Event):
  - Siklus 1: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)
  - Siklus 2: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)
  - Siklus 3: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)

Hari 2026-04-03 (3 kali Pemulihan/Event):
  - Siklus 1: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)
  - Siklus 2: Gagal (butuh minimal durasi log 2 baris pasca pompa mati)
  - Siklus 3: Anomali Logaritmik (Kurva tidak mengikuti sifat pemompaan normal)

Hari 2026-04-04 (3 kali Pemulihan/Event):
  - Siklus 1: Gagal (butuh minimal durasi log 2 baris pasca pompa mati)
  - Siklus 2: Anomali Logaritmik (Kurva tidak mengikuti sifat pem

In [22]:

# ============================================================
# CELL: DETEKSI SIKLUS POMPA-RECOVERY AKUIFER JIAT
# Sumber data: df_plot (sudah didefinisikan di cell sebelumnya)
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==============================================================================
# ⚙️ KONFIGURASI
# ==============================================================================
THRESHOLD_DELTA = 0.003   # m/jam: minimum delta TMA dianggap "bergerak"
MIN_DURASI_JAM  = 2       # jam: segmen lebih pendek dari ini = noise, dibuang

# ==============================================================================
# STEP 1: AMBIL KOLOM & HITUNG DELTA (dari df_plot)
# ==============================================================================
kolom_tma = [col for col in df_plot.columns if 'rerata' in col.lower()][0]
seri_tma  = df_plot[kolom_tma].copy()
delta     = seri_tma.diff().fillna(0)

# Label fase per titik
kondisi_pump = delta < -THRESHOLD_DELTA
kondisi_recv = delta >  THRESHOLD_DELTA
fase_labels  = pd.Series('STABIL', index=seri_tma.index)
fase_labels[kondisi_pump] = 'PUMPING'
fase_labels[kondisi_recv] = 'RECOVERY'

# ==============================================================================
# STEP 2: RUN-LENGTH ENCODING — Kelompokkan fase berurutan jadi segmen
# ==============================================================================
perubahan_fase = (fase_labels != fase_labels.shift()).cumsum()

df_seg = (
    pd.DataFrame({'fase': fase_labels, 'tma': seri_tma})
    .groupby(perubahan_fase, sort=False)
    .agg(
        fase       = ('fase', 'first'),
        ts_start   = ('fase', lambda x: x.index[0]),
        ts_end     = ('fase', lambda x: x.index[-1]),
        durasi_jam = ('fase', 'count'),
        tma_start  = ('tma',  'first'),
        tma_end    = ('tma',  'last'),
        tma_min    = ('tma',  'min'),
        tma_max    = ('tma',  'max'),
    )
    .reset_index(drop=True)
)

# Buang segmen noise (terlalu pendek)
df_seg = df_seg[df_seg['durasi_jam'] >= MIN_DURASI_JAM].reset_index(drop=True)

# ==============================================================================
# STEP 3: DETEKSI SIKLUS — Pasangan PUMPING → RECOVERY
# ==============================================================================
siklus_list = []
i = 0
while i < len(df_seg) - 1:
    if df_seg.at[i, 'fase'] == 'PUMPING' and df_seg.at[i+1, 'fase'] == 'RECOVERY':
        pump = df_seg.iloc[i]
        recv = df_seg.iloc[i + 1]

        drawdown_m   = pump['tma_start'] - pump['tma_end']
        delta_pulih  = recv['tma_end'] - recv['tma_start']
        
        persen_pulih = min(
            round(delta_pulih / drawdown_m * 100, 1) if drawdown_m > 0 else 0.0,
            100.0
        )

        siklus_list.append({
            'siklus_ke'           : len(siklus_list) + 1,
            'pompa_nyala'         : pump['ts_start'],
            'pompa_mati'          : pump['ts_end'],
            'recovery_selesai'    : recv['ts_end'],
            'durasi_pumping_jam'  : int(pump['durasi_jam']),
            'durasi_recovery_jam' : int(recv['durasi_jam']),
            'kedalaman_awal_m'    : round(float(pump['tma_start']), 4),
            'kedalaman_puncak_m'  : round(float(pump['tma_max']), 4),
            'kedalaman_akhir_m'   : round(float(recv['tma_end']), 4),
            'drawdown_m'          : round(float(drawdown_m), 4),
            'persen_pulih'        : persen_pulih,
        })
        i += 2
    else:
        i += 1

df_siklus = pd.DataFrame(siklus_list)

# ==============================================================================
# STEP 4: RINGKASAN STATISTIK
# ==============================================================================
print("=" * 65)
print("  📊 HASIL DETEKSI SIKLUS POMPA-RECOVERY  —  JIAT Pondok Kahuru")
print("=" * 65)
print(f"  ✅ Total Siklus Terdeteksi  : {len(df_siklus)} siklus")
if len(df_siklus) > 0:
    print(f"  ⏱  Rata² Durasi Pompa       : {df_siklus['durasi_pumping_jam'].mean():.1f} jam")
    print(f"  ⏱  Rata² Durasi Recovery    : {df_siklus['durasi_recovery_jam'].mean():.1f} jam")
    print(f"  📉 Rata² Drawdown           : {df_siklus['drawdown_m'].mean():.3f} m")
    print(f"  💧 Rata² Persen Pulih       : {df_siklus['persen_pulih'].mean():.1f}%")
    anomali = df_siklus[df_siklus['persen_pulih'] < 80]
    print(f"  ⚠️  Siklus Anomali (<80% pulih): {len(anomali)} siklus")
print("=" * 65)
display(df_siklus)

# ==============================================================================
# STEP 5: VISUALISASI DUAL-PANEL
# Panel Atas : TMA series + shading merah (pompa) & kuning (recovery) per siklus
# Panel Bawah: Bar chart durasi pompa vs recovery per siklus
# ==============================================================================
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=(
        '<b>Kurva TMA + Segmentasi Siklus Terdeteksi</b>',
        '<b>Durasi Tiap Siklus: Pumping (Merah) vs Recovery (Kuning)</b>'
    ),
    vertical_spacing=0.08
)

# ── PANEL 1: Garis TMA utama ──
fig.add_trace(
    go.Scatter(
        x=seri_tma.index,
        y=seri_tma.values,
        name='TMA (Rerata)',
        mode='lines',
        line=dict(color='rgba(148, 163, 184, 0.6)', width=1.5),
    ),
    row=1, col=1
)

# ── PANEL 1: Shading per siklus ──
for _, sk in df_siklus.iterrows():
    n = int(sk['siklus_ke'])

    # Zona PUMPING → merah transparan
    fig.add_vrect(
        x0=sk['pompa_nyala'], x1=sk['pompa_mati'],
        fillcolor='rgba(239, 68, 68, 0.15)', line_width=0,
        row=1, col=1
    )
    # Zona RECOVERY → kuning transparan
    fig.add_vrect(
        x0=sk['pompa_mati'], x1=sk['recovery_selesai'],
        fillcolor='rgba(250, 204, 21, 0.12)', line_width=0,
        row=1, col=1
    )
    # Label nomor siklus di atas zona pumping
    fig.add_annotation(
        x=sk['pompa_nyala'],
        y=seri_tma.max(),
        text=f"S{n}",
        showarrow=False,
        font=dict(color='#ef4444', size=10, family='monospace'),
        xanchor='left',
        row=1, col=1
    )

# ── PANEL 1: Marker titik pompa ON/OFF ──
if len(df_siklus) > 0:
    fig.add_trace(
        go.Scatter(
            x=df_siklus['pompa_nyala'],
            y=df_siklus['kedalaman_awal_m'],
            name='Pompa ON',
            mode='markers',
            marker=dict(color='#ef4444', size=10, symbol='triangle-down',
                        line=dict(color='white', width=1)),
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=df_siklus['recovery_selesai'],
            y=df_siklus['kedalaman_akhir_m'],
            name='Recovery ✓',
            mode='markers',
            marker=dict(color='#22c55e', size=10, symbol='circle',
                        line=dict(color='white', width=1)),
        ),
        row=1, col=1
    )

# ── PANEL 2: Bar chart durasi ──
if len(df_siklus) > 0:
    label_siklus = [f"S{int(x)}" for x in df_siklus['siklus_ke']]

    fig.add_trace(
        go.Bar(
            x=label_siklus,
            y=df_siklus['durasi_pumping_jam'],
            name='Durasi Pompa (jam)',
            marker_color='#ef4444',
            marker_line_color='rgba(0,0,0,0.3)',
            marker_line_width=0.5,
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Bar(
            x=label_siklus,
            y=df_siklus['durasi_recovery_jam'],
            name='Durasi Recovery (jam)',
            marker_color='#facc15',
            marker_line_color='rgba(0,0,0,0.3)',
            marker_line_width=0.5,
        ),
        row=2, col=1
    )

# ==============================================================================
# KOSMETIKA
# ==============================================================================
fig.update_layout(
    title='<b>Analisis Deteksi Siklus Akuifer JIAT — Pumping & Recovery</b>',
    template='plotly_dark',
    hovermode='x unified',
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=700,
)

fig.update_yaxes(
    title_text='Kedalaman TMA (m)',
    autorange='reversed',
    row=1, col=1,
    color='#94a3b8',
    showgrid=False,
)
fig.update_yaxes(
    title_text='Jam',
    row=2, col=1,
    color='#94a3b8',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.07)',
)
fig.update_xaxes(
    dtick='43200000',
    tickformat='%d %b\n%H:%M',
    row=1, col=1,
)

fig.show()



  📊 HASIL DETEKSI SIKLUS POMPA-RECOVERY  —  JIAT Pondok Kahuru
  ✅ Total Siklus Terdeteksi  : 22 siklus
  ⏱  Rata² Durasi Pompa       : 6.2 jam
  ⏱  Rata² Durasi Recovery    : 3.1 jam
  📉 Rata² Drawdown           : 0.082 m
  💧 Rata² Persen Pulih       : 86.4%
  ⚠️  Siklus Anomali (<80% pulih): 7 siklus


,siklus_ke,pompa_nyala,pompa_mati,recovery_selesai,durasi_pumping_jam,durasi_recovery_jam,kedalaman_awal_m,kedalaman_puncak_m,kedalaman_akhir_m,drawdown_m,persen_pulih
0,1,2026-04-01 01:00:00,2026-04-01 04:00:00,2026-04-01 08:00:00,4,3,17.811,17.811,17.946,0.021,100.0
1,2,2026-04-01 09:00:00,2026-04-01 15:00:00,2026-04-01 18:00:00,7,3,17.926,17.926,17.942,0.146,76.7
2,3,2026-04-01 19:00:00,2026-04-02 03:00:00,2026-04-02 08:00:00,9,4,17.934,17.934,17.980,0.133,100.0
3,4,2026-04-02 09:00:00,2026-04-02 12:00:00,2026-04-02 14:00:00,4,2,17.951,17.951,17.869,0.107,7.5
4,5,2026-04-02 19:00:00,2026-04-03 03:00:00,2026-04-03 08:00:00,9,4,17.907,17.907,17.997,0.085,100.0
5,6,2026-04-03 09:00:00,2026-04-03 12:00:00,2026-04-03 18:00:00,4,3,17.951,17.951,17.977,0.092,100.0
6,7,2026-04-03 19:00:00,2026-04-04 03:00:00,2026-04-04 07:00:00,9,2,17.969,17.969,17.988,0.113,51.3
7,8,2026-04-04 09:00:00,2026-04-04 15:00:00,2026-04-04 19:00:00,7,3,17.942,17.942,18.018,0.072,100.0
8,9,2026-04-04 20:00:00,2026-04-05 03:00:00,2026-04-05 08:00:00,8,4,17.963,17.963,18.048,0.092,100.0
9,10,2026-04-05 20:00:00,2026-04-06 02:00:00,2026-04-06 07:00:00,7,3,17.976,17.976,18.082,0.055,100.0
